# 01 - Veri Temizleme (Rossmann Mağaza Satışları)

**Amaç:** Ham Kaggle dosyalarını (`train.csv`, `store.csv`, `test.csv`) analize ve tahmine hazır hale getirmek.

**İlke:** Hiçbir satırı sessizce silmiyoruz. Her temizlik kararının nedenini ve etkilediği satır sayısını
`temizlik_ozeti.csv` dosyasına yazıyoruz. Jüri sunumunda "ne yaptın, neden yaptın, kaç satırı etkiledi" sorusunun cevabı bu dosya.

**Klasörler:** Ham dosyalar `ham/` klasöründe, temiz çıktılar `temiz/` klasörüne yazılır.

In [ ]:
import os
import numpy as np
import pandas as pd

VERI_KLASORU = "ham"
CIKTI_KLASORU = "temiz"
os.makedirs(CIKTI_KLASORU, exist_ok=True)

ozet = []  # Her temizlik adımının kaydı burada tutulur


def kaydet(tablo, sorun, adet, karar):
    ozet.append({"Tablo": tablo, "Sorun": sorun, "Etkilenen_Satir_veya_Magaza": int(adet), "Karar": karar})
    print(f"[{tablo}] {sorun}: {int(adet):,} -> {karar}")

## 1. Veriyi yükleme
`StateHoliday` sütununda hem sayı (0) hem harf (a, b, c) olduğu için onu metin olarak okuyoruz.

In [ ]:
train = pd.read_csv(f"{VERI_KLASORU}/train.csv", low_memory=False, parse_dates=["Date"], dtype={"StateHoliday": "str"})
store = pd.read_csv(f"{VERI_KLASORU}/store.csv")
test = pd.read_csv(f"{VERI_KLASORU}/test.csv", parse_dates=["Date"], dtype={"StateHoliday": "str"})

ham_boyut = {"train": train.shape, "store": store.shape, "test": test.shape}
train_ham = train.copy()  # test tablosundaki eksikleri doldururken eski davranışa bakacağız
print(ham_boyut)

## 2. Train tablosu: günlük satış kayıtları

### 2.1 Yapısal kontroller (sorun çıkmazsa hiçbir şey yapılmaz)
- Aynı mağaza ve aynı gün için tekrar eden satır var mı?
- `DayOfWeek` sütunu tarihle uyumlu mu?
- Mağaza kapalıyken (Open=0) satış kaydı var mı?

In [ ]:
tekrar = train.duplicated(["Store", "Date"]).sum()
gun_uyumsuz = ((train.Date.dt.dayofweek + 1) != train.DayOfWeek).sum()
kapali_satisli = ((train.Open == 0) & (train.Sales > 0)).sum()
negatif = (train.Sales < 0).sum()
gecerli_tatil = set(train.StateHoliday.unique()) <= {"0", "a", "b", "c"}

kaydet("train", "Aynı mağaza-gün tekrar eden satır", tekrar, "Sorun yok, işlem yapılmadı")
kaydet("train", "DayOfWeek ile tarih uyumsuz", gun_uyumsuz, "Sorun yok, işlem yapılmadı")
kaydet("train", "Kapalı gün (Open=0) ama satış>0", kapali_satisli, "Sorun yok, işlem yapılmadı")
kaydet("train", "Negatif satış", negatif, "Sorun yok, işlem yapılmadı")
assert tekrar == 0 and gun_uyumsuz == 0 and kapali_satisli == 0 and negatif == 0 and gecerli_tatil

### 2.2 Eksik günler (kayıt hiç yok)
1.115 mağaza x 942 gün olmalıydı. Eksik kayıtların hangi mağazalarda ve hangi tarihlerde olduğuna bakıyoruz.
Bu günlerin satışını **uydurmuyoruz** (doldurmuyoruz). Sadece raporluyoruz, zaman serisi aşamasında bilinçli ele alacağız.

In [ ]:
tum_tarihler = pd.date_range(train.Date.min(), train.Date.max())
beklenen = pd.MultiIndex.from_product([sorted(train.Store.unique()), tum_tarihler], names=["Store", "Date"])
mevcut = pd.MultiIndex.from_frame(train[["Store", "Date"]])
eksik = beklenen.difference(mevcut).to_frame(index=False)

eksik_ozet = eksik.groupby("Store").Date.agg(Eksik_Gun_Sayisi="count", Ilk_Eksik_Tarih="min", Son_Eksik_Tarih="max").reset_index()
eksik_ozet.to_csv(f"{CIKTI_KLASORU}/eksik_gunler_ozeti.csv", index=False)

yarim_donem = eksik_ozet[(eksik_ozet.Ilk_Eksik_Tarih == "2014-07-01") & (eksik_ozet.Son_Eksik_Tarih == "2014-12-31")]
print("Eksik günü olan mağaza:", len(eksik_ozet), "| toplam eksik gün:", len(eksik))
print("2014-07-01 / 2014-12-31 arası tamamen kayıtsız mağaza:", len(yarim_donem))
kaydet("train", "Eksik tarih kaydı (mağaza-gün)", len(eksik),
       f"Doldurulmadı. {len(yarim_donem)} mağazada 2014 Temmuz-Aralık kaydı yok; eksik_gunler_ozeti.csv'ye yazıldı")

### 2.3 Açık görünüp satışı sıfır olan günler
`Open=1` ama `Sales=0` olması mantıksız (mağaza açıksa satış olmalı). Bunlar veri girişi hatası veya kayıt eksikliği.
**Karar:** Bu satırları siliyoruz.

In [ ]:
maske = (train.Open == 1) & (train.Sales == 0)
kaydet("train", "Open=1 ama Sales=0", maske.sum(), "Satırlar silindi (mantıksız kayıt)")
train = train[~maske]

### 2.4 Kapalı mağaza günleri
Mağaza kapalıyken satış her zaman 0'dır. Bu günler satış tahminini bozar (modelin "bu gün neden satış yok?" diye karışmasına yol açar).
Kapalı günler mağaza tarafından önceden bilinir, yani stok planlamasında zaten tahmin edilecek bir şey değildir.
**Karar:** Modelleme tablosundan çıkarıyoruz. (Ham dosya `ham/train.csv` olduğu gibi duruyor, bilgi kaybı yok.)

In [ ]:
maske = train.Open == 0
kaydet("train", "Mağaza kapalı gün (Open=0, Sales=0)", maske.sum(), "Satırlar modelleme tablosundan çıkarıldı")
train = train[~maske].drop(columns="Open")  # artık hepsi 1, sütun bilgi taşımıyor

### 2.5 Aykırı (uç) değerler
Her mağazanın kendi satış dağılımına göre, çeyrekler arası genişliğin (IQR) 3 katından uzak günleri **işaretliyoruz** (silmiyoruz).
Neden silmiyoruz? Çoğu promosyon ve tatil öncesi gerçek satış artışı. Bunlar tam da ölçmek istediğimiz etkiler.

In [ ]:
g = train.groupby("Store").Sales
q1 = train.Store.map(g.quantile(0.25))
q3 = train.Store.map(g.quantile(0.75))
iqr = q3 - q1
train["Sales_Aykiri"] = ((train.Sales > q3 + 3 * iqr) | (train.Sales < q1 - 3 * iqr)).astype(int)
kaydet("train", "Mağaza içi aykırı satış (3xIQR dışı)", train.Sales_Aykiri.sum(), "Silinmedi, Sales_Aykiri=1 ile işaretlendi")

### 2.6 Tarih parçaları
SQL ve grafik aşamalarında işimize yarayacak yıl, ay ve hafta sütunları.

In [ ]:
train["Year"] = train.Date.dt.year
train["Month"] = train.Date.dt.month
train["WeekOfYear"] = train.Date.dt.isocalendar().week.astype(int)
train = train.sort_values(["Store", "Date"]).reset_index(drop=True)

## 3. Store tablosu: mağaza bilgileri

### 3.1 Rekabet mesafesi (3 mağazada eksik)
Medyan ile dolduruyoruz (uç değerlerden etkilenmez) ve hangi mağazaların doldurulduğunu `CompetitionDistance_Eksikti` ile işaretliyoruz.

In [ ]:
store["CompetitionDistance_Eksikti"] = store.CompetitionDistance.isna().astype(int)
medyan = store.CompetitionDistance.median()
kaydet("store", "CompetitionDistance eksik", store.CompetitionDistance.isna().sum(), f"Medyan ({medyan:.0f} m) ile dolduruldu, işaretlendi")
store["CompetitionDistance"] = store.CompetitionDistance.fillna(medyan)

### 3.2 Rakibin açılış tarihi
- Mantıksız yıllar (örn. 1900) hatalı girişdir. 1950'den önceki yılları "bilinmiyor" yapıyoruz.
- Açılış tarihi bilinmeyen mağazaları uydurma bir tarihle **doldurmuyoruz**. `CompetitionOpenSince_Bilinmiyor` ile işaretliyoruz.

In [ ]:
gecersiz = store.CompetitionOpenSinceYear < 1950
kaydet("store", "Rakip açılış yılı mantıksız (<1950)", gecersiz.sum(), "Boş (bilinmiyor) yapıldı")
store.loc[gecersiz, ["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]] = np.nan

store["CompetitionOpenSince_Bilinmiyor"] = store.CompetitionOpenSinceYear.isna().astype(int)
kaydet("store", "Rakip açılış ay/yıl bilgisi boş", store.CompetitionOpenSince_Bilinmiyor.sum(), "Doldurulmadı, işaretlendi")

### 3.3 Promo2 (uzun dönem promosyon) sütunları
`Promo2=0` olan mağazalar bu promosyona hiç katılmıyor, dolayısıyla başlangıç hafta/yıl ve aralık bilgisi **olması gerekmez**.
Bu eksikler hata değil. Okunabilirlik için: aralığı "Yok", hafta ve yılı 0 yapıyoruz.

Ayrıca `PromoInterval` içinde "Sept" yazılmış, diğer aylar 3 harfli. Tutarlılık için "Sep" yapıyoruz.

In [ ]:
promo2_yok = (store.Promo2 == 0).sum()
kaydet("store", "Promo2 yok, bu yüzden hafta/yıl/aralık boş (yapısal eksik)", promo2_yok, "Aralık='Yok', hafta/yıl=0")
store["PromoInterval"] = store.PromoInterval.fillna("Yok")
store[["Promo2SinceWeek", "Promo2SinceYear"]] = store[["Promo2SinceWeek", "Promo2SinceYear"]].fillna(0).astype(int)

sept = store.PromoInterval.str.contains("Sept").sum()
kaydet("store", "PromoInterval'da 'Sept' yazımı", sept, "'Sep' olarak düzeltildi")
store["PromoInterval"] = store.PromoInterval.str.replace("Sept", "Sep", regex=False)

## 4. Test tablosu
`Open` sütununda 11 eksik var (hepsi aynı mağaza). Mağazanın geçmişteki haftanın günü davranışına bakıp dolduruyoruz:
o mağaza o gün geçmişte çoğunlukla açıksa 1, değilse 0.

In [ ]:
desen = (train_ham.groupby(["Store", "DayOfWeek"]).Open.mean() >= 0.5).astype(int).rename("Open_Tahmin").reset_index()
eksik_open = test.Open.isna()
kaydet("test", "Open eksik", eksik_open.sum(), "Mağazanın geçmiş haftanın günü davranışına göre dolduruldu")
test = test.merge(desen, on=["Store", "DayOfWeek"], how="left")
test["Open"] = test.Open.fillna(test.Open_Tahmin).astype(int)
test = test.drop(columns="Open_Tahmin")

## 5. Tabloların birleştirilebildiğini kontrol et
Her satış kaydının bir mağaza bilgisi karşılığı olmalı (birleştirince satır sayısı değişmemeli, eksik çıkmamalı).

In [ ]:
birlesik = train.merge(store, on="Store", how="left", validate="m:1")
assert len(birlesik) == len(train), "Birleştirme satır sayısını değiştirdi!"
assert birlesik.CompetitionDistance.isna().sum() == 0, "Eşleşmeyen mağaza var!"
print("Birleştirme başarılı:", birlesik.shape)

## 6. Temiz dosyaları kaydet ve önce/sonra özetini yaz

In [ ]:
train.to_csv(f"{CIKTI_KLASORU}/train_temiz.csv", index=False)
store.to_csv(f"{CIKTI_KLASORU}/store_temiz.csv", index=False)
test.to_csv(f"{CIKTI_KLASORU}/test_temiz.csv", index=False)
pd.DataFrame(ozet).to_csv(f"{CIKTI_KLASORU}/temizlik_ozeti.csv", index=False, encoding="utf-8-sig")

son_boyut = {"train": train.shape, "store": store.shape, "test": test.shape}
karsilastirma = pd.DataFrame({"Once": ham_boyut, "Sonra": son_boyut}).T
print(karsilastirma)
print("Temiz train eksik değer:", int(train.isna().sum().sum()), "| store eksik (rakip açılış tarihi hariç):",
      int(store.drop(columns=["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]).isna().sum().sum()),
      "| test eksik:", int(test.isna().sum().sum()))